# 04 — Preprocesado y diseño del target

**Proyecto:** VIGÍA (Sentry) — TFC
**Objetivo:** Convertir el dataset limpio en un dataset listo para entrenar:
diseño del target (binario + familias), transformaciones, split y guardado en Parquet.

**Entrada:** `cic-ids2017-parquet-clean` (2.830.743 filas × 79 columnas).

**Decisiones que se toman en este notebook:**
- Target binario (0=BENIGN, 1=ATAQUE) como baseline principal.
- Target por familias (7 clases) para la Fase 2.
- Agrupación de clases minoritarias en una familia "Otros".
- Split estratificado 70/15/15 con semilla fija (reproducibilidad).
- Exclusión de `Label` original del conjunto de features.

In [1]:
# ============================================
# Setup: repo, drive, Spark, carga del Parquet clean
# ============================================
import os
from google.colab import drive

GITHUB_USER = "AguCS231"
REPO_NAME   = "sentry"
REPO_LOCAL  = f"/content/{REPO_NAME}"
PARQUET_CLEAN = "/content/drive/MyDrive/cic-ids2017-parquet-clean"

# Clonar repo si no existe
if not os.path.exists(REPO_LOCAL):
    print("Clonando repo...")
    %cd /content
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
    %cd {REPO_LOCAL}
    !git config --global user.email "srcasares2@gmail.com"
    !git config --global user.name "{GITHUB_USER}"
else:
    %cd {REPO_LOCAL}
    print(f"✅ Repo: {REPO_LOCAL}")

drive.mount('/content/drive')

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("VIGIA_Preprocesado") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"\n✅ Spark {spark.version}")

# Cargar dataset limpio
df = spark.read.parquet(PARQUET_CLEAN)
df.cache()

print(f"\n📥 Dataset cargado desde: {PARQUET_CLEAN}")
print(f"   Filas:    {df.count():,}")
print(f"   Columnas: {len(df.columns)}")

Clonando repo...
/content
Cloning into 'sentry'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 72 (delta 25), reused 28 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 24.50 KiB | 1.11 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/content/sentry
Mounted at /content/drive

✅ Spark 4.0.4

📥 Dataset cargado desde: /content/drive/MyDrive/cic-ids2017-parquet-clean
   Filas:    2,830,743
   Columnas: 79


## 1. Diseño del target binario

Traducimos `Label` a una columna numérica `target_bin`:
- `0` = BENIGN (tráfico legítimo)
- `1` = ATAQUE (cualquier tipo de intrusión)

**Motivo de seguridad:** este es el escenario operativo de un IDS. El analista
no necesita saber "qué tipo de DoS es", necesita saber "¿esto es ataque o no?".

**Limitación conocida:** perdemos granularidad. Un modelo binario detectará
cualquier ataque, pero no dirá cuál. La familia la resolvemos en Fase 2.

In [2]:
# ============================================
# Crear target binario
# ============================================
from pyspark.sql.functions import col, when

df = df.withColumn(
    "target_bin",
    when(col("Label") == "BENIGN", 0).otherwise(1)
)

# Verificación: distribución del target
print("=== Distribución del target binario ===\n")
df.groupBy("target_bin").count().orderBy("target_bin").show()

# Equivalencia: ¿la suma coincide con el total?
total = df.count()
benign = df.filter(col("target_bin") == 0).count()
ataque = df.filter(col("target_bin") == 1).count()

print(f"\nTotal:   {total:,}")
print(f"BENIGN:  {benign:,}  ({benign/total*100:.2f}%)")
print(f"ATAQUE:  {ataque:,}  ({ataque/total*100:.2f}%)")
print(f"Ratio:   {benign/ataque:.2f}:1")

=== Distribución del target binario ===

+----------+-------+
|target_bin|  count|
+----------+-------+
|         0|2273097|
|         1| 557646|
+----------+-------+


Total:   2,830,743
BENIGN:  2,273,097  (80.30%)
ATAQUE:  557,646  (19.70%)
Ratio:   4.08:1


## 2. Target por familias

Agrupamos las 15 clases originales en 8 familias operativas. Esto reduce el
ruido y refleja la granularidad que un analista de SOC necesita: no le importa
si el DoS es Hulk o GoldenEye, le importa que es un DoS.

**Motivo de seguridad:** las clases con muy pocas muestras (Heartbleed: 11,
Infiltration: 36, Sql Injection: 21) no son viables por separado. Agruparlas
en "Otros" permite medir si el modelo generaliza a ataques no vistos.

**Esta columna se usará en Fase 2.** En Fase 1 (binario) no se toca.

In [3]:
# ============================================
# Crear target por familias
# ============================================
from pyspark.sql.functions import col, when

df = df.withColumn(
    "target_family",
    when(col("Label") == "BENIGN", "BENIGN")
    .when(col("Label").isin("DoS Hulk", "DoS GoldenEye",
                            "DoS slowloris", "DoS Slowhttptest"), "DoS")
    .when(col("Label") == "PortScan", "Probe")
    .when(col("Label") == "DDoS", "DDoS")
    .when(col("Label").isin("FTP-Patator", "SSH-Patator"), "Brute-force")
    .when(col("Label").isin("Web Attack - Brute Force",
                            "Web Attack - XSS",
                            "Web Attack - Sql Injection"), "Web")
    .when(col("Label") == "Bot", "Bot")
    .otherwise("Otros")   # Infiltration, Heartbleed
)

print("✅ target_family creado")

✅ target_family creado


In [4]:
# ============================================
# Verificación: distribución por familia
# ============================================
from pyspark.sql.functions import col, desc, round as spark_round

print("=== Distribución por familia ===\n")
total = df.count()

fam = df.groupBy("target_family").count() \
    .orderBy(desc("count")) \
    .withColumn("porcentaje", spark_round(col("count") / total * 100, 4))

fam.show(truncate=False)

# Cruce: ¿cada familia agrupa bien las clases originales?
print("\n=== Cruce Label × Familia ===\n")
cruce = df.groupBy("target_family", "Label").count().orderBy("target_family", "Label")
cruce.show(n=30, truncate=False)

=== Distribución por familia ===

+-------------+-------+----------+
|target_family|count  |porcentaje|
+-------------+-------+----------+
|BENIGN       |2273097|80.3004   |
|DoS          |252661 |8.9256    |
|Probe        |158930 |5.6144    |
|DDoS         |128027 |4.5227    |
|Brute-force  |13835  |0.4887    |
|Web          |2180   |0.077     |
|Bot          |1966   |0.0695    |
|Otros        |47     |0.0017    |
+-------------+-------+----------+


=== Cruce Label × Familia ===

+-------------+--------------------------+-------+
|target_family|Label                     |count  |
+-------------+--------------------------+-------+
|BENIGN       |BENIGN                    |2273097|
|Bot          |Bot                       |1966   |
|Brute-force  |FTP-Patator               |7938   |
|Brute-force  |SSH-Patator               |5897   |
|DDoS         |DDoS                      |128027 |
|DoS          |DoS GoldenEye             |10293  |
|DoS          |DoS Hulk                  |231073 |
|Do

## 3. Definición del conjunto de features

Separamos el dataset en:
- **Features**: las 77 columnas numéricas que el modelo puede usar.
- **Metadata**: `Label` (texto), `target_bin`, `target_family` (targets, no features).

**Motivo de seguridad:** ninguna columna que contenga la respuesta puede
entrar como feature. Es la regla número 1 del ML en seguridad.

In [5]:
# ============================================
# Definir feature_cols y columnas excluidas
# ============================================
# Excluimos: Label (texto), target_bin, target_family (targets)
excluir = {"Label", "target_bin", "target_family"}

feature_cols = [c for c in df.columns if c not in excluir]
print(f"Features: {len(feature_cols)}")
print(f"Excluidas: {sorted(excluir)}")

# Verificar tipos (todas deben ser numéricas)
from pyspark.sql.types import NumericType
no_numericas = [c for c in feature_cols
                if not isinstance(df.schema[c].dataType, NumericType)]
if no_numericas:
    print(f"\n⚠️ Features no numéricas: {no_numericas}")
else:
    print("\n✅ Todas las features son numéricas")

Features: 78
Excluidas: ['Label', 'target_bin', 'target_family']

✅ Todas las features son numéricas


## 4. Transformación log1p de features de cola larga

Muchas features tienen distribuciones muy sesgadas: máximo de 2×10⁹ frente a
una mediana de 10³. Esto es un problema porque:

1. Los árboles (XGBoost) se las apañan bien, pero **empeoran** con colas extremas.
2. Las redes neuronales o SVM se rompen.

**Solución:** aplicar `log1p(x) = log(1+x)`. Comprime los valores grandes y
deja los pequeños casi igual. La diferencia relativa entre 10 y 100 se
mantiene; la diferencia entre 10⁸ y 10⁹ se reduce.

**Motivo de seguridad:** un atacante puede inflar `Flow_Bytes/s` artificialmente
(un solo flujo de 2 GB/s). Si no comprimimos, esa única feature domina la
predicción y el modelo se vuelve trivialmente evadible.

**Criterio:** aplicamos log1p a columnas donde `max/mediana > 1000`.

In [6]:
# ============================================
# Identificar features con cola larga y aplicar log1p
# ============================================
from pyspark.sql.functions import col, log1p

# Columnas donde ya sabemos que hay cola larga (visto en el EDA)
# Filtramos por ratio max/mediana para ser sistemáticos
print("=== Analizando skewness (max/mediana) ===\n")

log_cols = []
for c in feature_cols:
    try:
        cuantiles = df.approxQuantile(c, [0.5, 1.0], 0.01)
        mediana = cuantiles[0]
        maximo = cuantiles[1]
        if mediana and mediana > 0 and maximo and maximo > 0:
            ratio = maximo / mediana
            if ratio > 1000:
                log_cols.append(c)
                print(f"  {c:<40} max/mediana = {ratio:>15,.1f}")
    except Exception as e:
        pass

print(f"\n✅ Columnas con cola larga (ratio > 1000): {len(log_cols)}")

# Aplicar log1p
for c in log_cols:
    df = df.withColumn(c, log1p(col(c)))

print(f"\n✅ log1p aplicado a {len(log_cols)} columnas")

=== Analizando skewness (max/mediana) ===

  Flow_Duration                            max/mediana =         3,840.5
  Total_Fwd_Packets                        max/mediana =       109,879.5
  Total_Backward_Packets                   max/mediana =       145,961.0
  Total_Length_of_Fwd_Packets              max/mediana =       208,064.5
  Total_Length_of_Bwd_Packets              max/mediana =     5,285,911.5
  Fwd_Packet_Length_Min                    max/mediana =         1,162.5
  Flow_Bytes/s                             max/mediana =       443,224.4
  Flow_Packets/s                           max/mediana =        38,701.0
  Flow_IAT_Mean                            max/mediana =        10,039.5
  Flow_IAT_Std                             max/mediana =       664,608.0
  Flow_IAT_Max                             max/mediana =         3,897.5
  Flow_IAT_Min                             max/mediana =    30,000,000.0
  Fwd_IAT_Total                            max/mediana =     5,454,545.5
  Fwd_IA

## 5. Split estratificado train/val/test

Dividimos el dataset en tres partes:
- **Train (70%)**: el modelo aprende.
- **Validation (15%)**: ajuste de hiperparámetros.
- **Test (15%)**: evaluación final. **No se toca hasta el final.**

**Estratificación por `target_family`**: mantiene la proporción de cada
familia en los tres conjuntos. Sin esto, Heartbleed (11 muestras) podría caer
entera en test y el modelo no la vería nunca.

**Decisión de seguridad:** la familia "Otros" (47 muestras) va **íntegra a
test**, no a train. Simula el escenario real "¿detecta el modelo ataques que
no ha visto nunca?". Es un experimento de generalización, no de memoria.

In [7]:
# ============================================
# Split estratificado por target_family
# Excepción: "Otros" va íntegro a test (ataques no vistos)
# ============================================
from functools import reduce
from pyspark.sql import DataFrame

familias = [r["target_family"] for r in df.select("target_family").distinct().collect()]
familias_entrenables = [f for f in familias if f != "Otros"]

print(f"Familias entrenables: {familias_entrenables}")
print(f"Familia reservada para test: Otros\n")

train_parts, val_parts, test_parts = [], [], []

for fam in familias_entrenables:
    subset = df.filter(col("target_family") == fam)
    tr, va, te = subset.randomSplit([0.7, 0.15, 0.15], seed=42)
    train_parts.append(tr)
    val_parts.append(va)
    test_parts.append(te)

# Añadir "Otros" completo a test
test_parts.append(df.filter(col("target_family") == "Otros"))

train_df = reduce(DataFrame.unionByName, train_parts)
val_df   = reduce(DataFrame.unionByName, val_parts)
test_df  = reduce(DataFrame.unionByName, test_parts)

print("✅ Split completado")

Familias entrenables: ['BENIGN', 'DoS', 'Brute-force', 'Bot', 'DDoS', 'Probe', 'Web']
Familia reservada para test: Otros

✅ Split completado


In [8]:
# ============================================
# Verificar split
# ============================================
from pyspark.sql.functions import col, desc

n_train = train_df.count()
n_val   = val_df.count()
n_test  = test_df.count()
n_total = n_train + n_val + n_test

print("=== Tamaños ===")
print(f"Train: {n_train:>10,}  ({n_train/n_total*100:.2f}%)")
print(f"Val:   {n_val:>10,}  ({n_val/n_total*100:.2f}%)")
print(f"Test:  {n_test:>10,}  ({n_test/n_total*100:.2f}%)")
print(f"Total: {n_total:>10,}")

print("\n=== Distribución por familia en cada split ===\n")
for nombre, d in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"--- {nombre} ---")
    d.groupBy("target_family").count().orderBy(desc("count")).show(truncate=False)

=== Tamaños ===
Train:  1,981,991  (70.02%)
Val:      424,173  (14.98%)
Test:     424,579  (15.00%)
Total:  2,830,743

=== Distribución por familia en cada split ===

--- Train ---
+-------------+-------+
|target_family|count  |
+-------------+-------+
|BENIGN       |1591607|
|DoS          |176744 |
|Probe        |111330 |
|DDoS         |89682  |
|Brute-force  |9742   |
|Web          |1520   |
|Bot          |1366   |
+-------------+-------+

--- Val ---
+-------------+------+
|target_family|count |
+-------------+------+
|BENIGN       |340643|
|DoS          |37870 |
|Probe        |23828 |
|DDoS         |19164 |
|Brute-force  |2054  |
|Web          |325   |
|Bot          |289   |
+-------------+------+

--- Test ---
+-------------+------+
|target_family|count |
+-------------+------+
|BENIGN       |340847|
|DoS          |38047 |
|Probe        |23772 |
|DDoS         |19181 |
|Brute-force  |2039  |
|Web          |335   |
|Bot          |311   |
|Otros        |47    |
+-------------+------+

## 6. Guardar los tres conjuntos en Parquet

Cada split se guarda como Parquet independiente:
- `train.parquet`
- `val.parquet`
- `test.parquet`

**Motivo:** el notebook 05 (modelado) solo carga estos tres ficheros. No repite
el preprocesado. Si el preprocesado cambia, solo se regeneran aquí.

**Motivo de seguridad:** tener el test separado físicamente evita que se
"cuele" en el entrenamiento por error. Es una barrera operativa, no solo lógica.

In [9]:
# ============================================
# Guardar splits en Parquet
# ============================================
BASE_OUT = "/content/drive/MyDrive/cic-ids2017-splits"

train_path = f"{BASE_OUT}/train.parquet"
val_path   = f"{BASE_OUT}/val.parquet"
test_path  = f"{BASE_OUT}/test.parquet"

print("Guardando splits (2-4 minutos)...\n")

train_df.write.mode("overwrite").parquet(train_path)
print(f"✅ Train: {train_path}")

val_df.write.mode("overwrite").parquet(val_path)
print(f"✅ Val:   {val_path}")

test_df.write.mode("overwrite").parquet(test_path)
print(f"✅ Test:  {test_path}")

Guardando splits (2-4 minutos)...

✅ Train: /content/drive/MyDrive/cic-ids2017-splits/train.parquet
✅ Val:   /content/drive/MyDrive/cic-ids2017-splits/val.parquet
✅ Test:  /content/drive/MyDrive/cic-ids2017-splits/test.parquet


In [10]:
# ============================================
# Verificar que los Parquet se leen correctamente
# ============================================
for nombre, ruta in [("Train", train_path), ("Val", val_path), ("Test", test_path)]:
    d = spark.read.parquet(ruta)
    print(f"{nombre:6s} → {d.count():>10,} filas | {len(d.columns)} columnas")

Train  →  1,981,991 filas | 81 columnas
Val    →    424,173 filas | 81 columnas
Test   →    424,579 filas | 81 columnas


## 7. Decisiones cerradas en este notebook

| Decisión | Motivo técnico | Motivo de seguridad | Limitación |
|---|---|---|---|
| Target binario | Baseline sólido | Escenario real de IDS | Pierde granularidad de tipo |
| Target por familias (8) | Reduce ruido | Granularidad operativa de SOC | "Otros" tiene solo 47 muestras |
| log1p en colas largas | Estabiliza el modelo | Evita que un flujo manipulado domine la predicción | Ninguna relevante |
| Split 70/15/15 estratificado | Mantiene proporciones | Reproducible con seed=42 | No es temporal (no hay timestamps) |
| "Otros" íntegro a test | Mide generalización | Simula ataques no vistos en producción | Solo 47 muestras → métrica ruidosa |
| test.parquet separado | Evita fugas accidentales | Barrera operativa | — |

In [11]:
# ============================================
# Diario y commit
# ============================================
%cd /content/sentry

from datetime import datetime
fecha = datetime.now().strftime("%Y-%m-%d")

entrada = f"""

---

## {fecha} — Preprocesado y diseño del target

**Qué probé:**
- Diseño del target binario (BENIGN=0, ATAQUE=1).
- Diseño del target por familias (8 clases operativas).
- Identificación y transformación log1p de features de cola larga.
- Split estratificado 70/15/15 con seed=42.
- Guardado de train/val/test como Parquet separados.

**Qué encontré:**
- Ratio binario: 4.08:1 (BENIGN vs ATAQUE).
- Ratio familias: 48.364:1 (BENIGN vs Otros).
- Familias entrenables: BENIGN, DoS, Probe, DDoS, Brute-force, Web, Bot.
- Familia reservada para test: Otros (Infiltration + Heartbleed, 47 muestras).

**Decisiones:**
- Binario como baseline principal (Fase 1).
- Familias como aporte (Fase 2).
- "Otros" íntegro en test → experimento de generalización a ataques no vistos.
- test.parquet separado físicamente del train y val.
- log1p aplicado a features con ratio max/mediana > 1000.

**Próximos pasos:**
- Notebook 05: modelado binario con XGBoost + experimento con/sin Destination_Port.
"""

with open("docs/diario.md", "a") as f:
    f.write(entrada)

print("✅ Diario actualizado")
!tail -30 docs/diario.md

!git add .
!git commit -m "Preprocesado: targets binario y familias, log1p, split estratificado"
!git push

/content/sentry
✅ Diario actualizado
- EDA (análisis exploratorio) del dataset.
- Distribución de clases, valores nulos, infinitos, negativos.


---

## 2026-09-28 — Preprocesado y diseño del target

**Qué probé:**
- Diseño del target binario (BENIGN=0, ATAQUE=1).
- Diseño del target por familias (8 clases operativas).
- Identificación y transformación log1p de features de cola larga.
- Split estratificado 70/15/15 con seed=42.
- Guardado de train/val/test como Parquet separados.

**Qué encontré:**
- Ratio binario: 4.08:1 (BENIGN vs ATAQUE).
- Ratio familias: 48.364:1 (BENIGN vs Otros).
- Familias entrenables: BENIGN, DoS, Probe, DDoS, Brute-force, Web, Bot.
- Familia reservada para test: Otros (Infiltration + Heartbleed, 47 muestras).

**Decisiones:**
- Binario como baseline principal (Fase 1).
- Familias como aporte (Fase 2).
- "Otros" íntegro en test → experimento de generalización a ataques no vistos.
- test.parquet separado físicamente del train y val.
- log1p aplicado a features 

In [13]:
from getpass import getpass
TOKEN = getpass("Token: ")
%cd /content/sentry
!git remote set-url origin https://{TOKEN}@github.com/AguCS231/sentry.git
!git push

Token: ··········
/content/sentry
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 1014 bytes | 1014.00 KiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/AguCS231/sentry.git
   d6df339..79b0d3b  main -> main
